# Milan, Sentinel-2 stack (5 dates), GHSL — standard train/test split

Adds the cloud-free 2018-12-08 date to the existing 4-date Milan stack (now
5 dates, 50 bands instead of 40), to test whether it changes the result.
Point locations and labels are reused from the existing 4-date stack's own
train/test split (`./output/milan/ghsl/stack`); the 50-band values are freshly sampled from
Earth Engine since no local table has them yet.

Tuning and the final fit use the SAME train partition the 4-date stack used (by point identity, via the split tag carried through extraction); a genuine holdout RMSE/MAE/R2/Bias is computed on the matching test partition. This isolates the one variable that changed -- 4 vs 5 dates -- as cleanly as possible.

Compare against the 4-date stack's own split-based result
(the S2 stack row in the Milan validation table).

## Cell 1 · Imports & configuration

In [1]:
import ee
from gee_init import init_gee

init_gee()

import json
import os
import time
import warnings

import geemap
import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

from s2_utils import build_composite, water_mask

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_JOBS       = -1
N_FOLDS      = 5
BLOCK_SIZES  = [500, 1000, 2000]
N_ITER_RF    = 40
LABEL_COL    = 'GHSL'

TARGET  = 'GHSL'
MODE    = 'split'
SRC_DIR = './output/milan/ghsl/stack'
STACK5_DATES = ['2018-03-23', '2018-06-16', '2018-08-05', '2018-10-24', '2018-12-08']

OUTPUT_DIR = './output/milan/ghsl/stack5_split'
os.makedirs(OUTPUT_DIR, exist_ok=True)
DRIVE_FOLDER = 'GEE_exports'

print(f'Milan, S2 stack-5, {TARGET} -- {MODE}')
print(f'dates: {STACK5_DATES}')
print(f'source points: {SRC_DIR}  ->  output: {OUTPUT_DIR}')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


GEE initialised (project: imd-vietnam).


Milan, S2 stack-5, GHSL -- split
dates: ['2018-03-23', '2018-06-16', '2018-08-05', '2018-10-24', '2018-12-08']
source points: ./output/milan/ghsl/stack  ->  output: ./output/milan/ghsl/stack5_split


---
## Cell 2 · AOI, 5-date stack composite, extract at the existing stack's point locations

In [2]:
from shapely.geometry import mapping

aoi_gdf  = gpd.read_file('./data/aoi_milan/milano_aoi.gpkg').to_crs('EPSG:4326')
aoi_geom = ee.Geometry(mapping(aoi_gdf.geometry.union_all()))
print(f'Milan AOI area: {aoi_geom.area(1).getInfo() / 1e6:,.0f} km2')

non_water_mask = water_mask(aoi_geom)

SELECTED_DATES = STACK5_DATES
pred_source_img, s2_proj, band_names = build_composite(
    'COPERNICUS/S2_SR_HARMONIZED', SELECTED_DATES, aoi_geom,
    non_water=non_water_mask, method='stack')
print(f'S2 stack-5 bands: {len(band_names)}  dates: {len(SELECTED_DATES)}')

# Reuse the 4-date stack's own point locations + labels (geometry and label
# only -- the 40 existing bands are NOT reused, since this is a fresh
# composite). 'split' tags which partition each point belonged to, so the
# split-mode notebook can reproduce the EXACT same train/test division the
# 4-date stack was scored on.
train_src = gpd.read_file(f'{SRC_DIR}/spatial_train_pts.gpkg')
test_src  = gpd.read_file(f'{SRC_DIR}/spatial_test_pts.gpkg')
train_src['split'] = 'train'
test_src['split']  = 'test'
pts = pd.concat([train_src[['geometry', LABEL_COL, 'split']],
                  test_src[['geometry', LABEL_COL, 'split']]], ignore_index=True)
pts = gpd.GeoDataFrame(pts, geometry='geometry', crs=train_src.crs).reset_index(drop=True)
print(f'Reusing {len(train_src)} train + {len(test_src)} test = {len(pts)} point locations from the 4-date stack')

fc_pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([row.geometry.x, row.geometry.y]),
               {LABEL_COL: float(row[LABEL_COL]), 'split': row['split'], 'pid': int(i)})
    for i, row in pts.iterrows()
])
sampled = pred_source_img.sampleRegions(
    collection=fc_pts, properties=[LABEL_COL, 'split', 'pid'],
    scale=10, projection=s2_proj, tileScale=4, geometries=True)

t0 = time.time()
gdf_all = geemap.ee_to_gdf(sampled)
print(f'extracted {len(gdf_all)} / {len(pts)} points in {time.time() - t0:.1f}s')
n_before = len(gdf_all)
gdf_all = gdf_all.dropna(subset=band_names).reset_index(drop=True)
print(f'after NaN drop: {len(gdf_all)} ({100 * len(gdf_all) / n_before:.1f}% retained)')

gdf_all.to_file(f'{OUTPUT_DIR}/samples_Milan_stack5_all.gpkg', driver='GPKG')

gdf_utm = gdf_all.to_crs('EPSG:32632')
coords_all = np.column_stack([gdf_utm.geometry.x, gdf_utm.geometry.y])
X_all = gdf_all[band_names].values.astype(np.float32)
y_all = gdf_all[LABEL_COL].values.astype(np.float32)
print(f'X shape: {X_all.shape} | y range: {y_all.min():.0f}-{y_all.max():.0f}%')

Milan AOI area: 4,455 km2
S2 stack-5 bands: 50  dates: 5
Reusing 2450 train + 998 test = 3448 point locations from the 4-date stack


extracted 3430 / 3448 points in 158.1s
after NaN drop: 3430 (100.0% retained)


X shape: (3430, 50) | y range: 0-100%


---
## Cell 3 · Spatial block helper functions

In [3]:
def assign_blocks(coords, block_size):
    bx = (coords[:, 0] // block_size).astype(int)
    by = (coords[:, 1] // block_size).astype(int)
    return np.array([f'{x}_{y}' for x, y in zip(bx, by)])

def blocks_to_folds(block_ids, n_folds, seed):
    rng    = np.random.RandomState(seed)
    unique = rng.permutation(np.unique(block_ids))
    bmap   = {b: i % n_folds for i, b in enumerate(unique)}
    return np.array([bmap[b] for b in block_ids])

def apply_buffer(train_coords, test_coords, buffer_m):
    if len(train_coords) == 0 or buffer_m == 0:
        return np.ones(len(test_coords), dtype=bool)
    dists, _ = cKDTree(train_coords).query(test_coords, k=1)
    return dists > buffer_m

def make_spatial_splits(point_folds, n_folds):
    all_idx = np.arange(len(point_folds))
    return [(all_idx[point_folds != f], all_idx[point_folds == f])
            for f in range(n_folds)]

print('Spatial block functions defined.')

Spatial block functions defined.


---
## Cell 3b · Split back into the 4-date stack's own train/test partition

In [4]:
train_gdf = gdf_all[gdf_all['split'] == 'train'].reset_index(drop=True)
test_gdf  = gdf_all[gdf_all['split'] == 'test'].reset_index(drop=True)
print(f'train={len(train_gdf)}  test={len(test_gdf)}  (4-date stack had '
      f'{len(train_src)}/{len(test_src)} before this composite\'s own NaN drop)')

X_train = train_gdf[band_names].values.astype(np.float32)
y_train = train_gdf[LABEL_COL].values.astype(np.float32)
X_test  = test_gdf[band_names].values.astype(np.float32)
y_test  = test_gdf[LABEL_COL].values.astype(np.float32)

train_utm = train_gdf.to_crs('EPSG:32632')
coords_train = np.column_stack([train_utm.geometry.x, train_utm.geometry.y])

train=2437  test=993  (4-date stack had 2450/998 before this composite's own NaN drop)


---
## Cell 4 · RF hyperparameter tuning, 5-fold spatial-block CV, on TRAIN only

In [5]:
PARAM_SPACE = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'max_samples':       [None, 0.7, 0.8, 0.9],
}

best_rmse_tune, best_search, best_block = np.inf, None, None
for bs in BLOCK_SIZES:
    bids = assign_blocks(coords_train, bs)
    pf = blocks_to_folds(bids, N_FOLDS, RANDOM_STATE)
    cv_splits = make_spatial_splits(pf, N_FOLDS)
    t0 = time.time()
    search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
        param_distributions=PARAM_SPACE, n_iter=N_ITER_RF, cv=cv_splits,
        scoring='neg_root_mean_squared_error', n_jobs=N_JOBS,
        random_state=RANDOM_STATE, refit=True)
    search.fit(X_train, y_train)
    cv_rmse = -search.best_score_
    print(f'{bs}m spatial CV -> RMSE={cv_rmse:.3f}  ({time.time() - t0:.1f}s)')
    if cv_rmse < best_rmse_tune:
        best_rmse_tune, best_search, best_block = cv_rmse, search, bs

best_params = best_search.best_params_
print(f'\nBEST: block={best_block}m  CV_RMSE={best_rmse_tune:.3f}')
print(f'params={best_params}')

500m spatial CV -> RMSE=18.614  (223.1s)


1000m spatial CV -> RMSE=18.593  (216.3s)


2000m spatial CV -> RMSE=18.326  (215.7s)

BEST: block=2000m  CV_RMSE=18.326
params={'n_estimators': 500, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_samples': None, 'max_features': 0.3, 'max_depth': 30}


---
## Cell 5 · Fit on train, evaluate the holdout, export the predicted raster

In [6]:
rf_fitted = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, **best_params)
rf_fitted.fit(X_train, y_train)
joblib.dump(rf_fitted, f'{OUTPUT_DIR}/best_model_RF_stack5_split.joblib')

y_pred_test = np.clip(rf_fitted.predict(X_test), 0, 100)
holdout = dict(
    RMSE=float(np.sqrt(mean_squared_error(y_test, y_pred_test))),
    MAE=float(mean_absolute_error(y_test, y_pred_test)),
    R2=float(r2_score(y_test, y_pred_test)),
    Bias=float(np.mean(y_test - y_pred_test)),   # observed - predicted
)
print(f'HOLDOUT (train-only model, scored on the held-out test partition):')
print(f'  RMSE={holdout["RMSE"]:.2f}  MAE={holdout["MAE"]:.2f}  '
      f'R2={holdout["R2"]:.3f}  Bias={holdout["Bias"]:+.2f}')

# Final model for raster export is ALSO trained on train only, matching
# 01b/01d's own "retrain best model on full TRAINING set" convention.
features = [ee.Feature(None, {**{b: float(row[b]) for b in band_names}, 'IMD': float(row[LABEL_COL])})
            for _, row in train_gdf.iterrows()]
train_fc = ee.FeatureCollection(features)

clf = (ee.Classifier.smileRandomForest(
           numberOfTrees=best_params.get('n_estimators', 200),
           minLeafPopulation=best_params.get('min_samples_leaf', 1),
           seed=RANDOM_STATE)
       .setOutputMode('REGRESSION')
       .train(features=train_fc, classProperty='IMD', inputProperties=band_names))

pred_img = (pred_source_img.classify(clf).rename('IMD_predicted')
            .clip(aoi_geom).updateMask(non_water_mask).clamp(0, 100))

fname = 'GHSL_predicted_RF_S2_stack5_split_Milan'
out_tif = f'{OUTPUT_DIR}/{fname}.tif'
if os.path.exists(out_tif):
    print(f'raster already present at {out_tif} -- skipping export.')
else:
    task = ee.batch.Export.image.toDrive(
        image=pred_img, description=fname, folder=DRIVE_FOLDER,
        fileNamePrefix=fname, region=aoi_geom, scale=10, crs='EPSG:32632',
        maxPixels=1e10, fileFormat='GeoTIFF')
    task.start()
    print(f'submitted batch export task (id={task.id})')
    state = 'READY'
    while state not in ('COMPLETED', 'FAILED', 'CANCELLED'):
        time.sleep(15)
        state = task.status()['state']
        print(f'  {state}')
    if state != 'COMPLETED':
        raise RuntimeError(f'export {state} -- {task.status()}')
    creds = ee.data.get_persistent_credentials()
    drive = gbuild('drive', 'v3', credentials=creds)
    resp = drive.files().list(q=f"name = '{fname}.tif' and trashed = false",
                               fields='files(id, createdTime)').execute()
    files = sorted(resp.get('files', []), key=lambda f: f['createdTime'], reverse=True)
    if not files:
        raise RuntimeError(f'export completed but {fname}.tif not found on Drive.')
    request = drive.files().get_media(fileId=files[0]['id'])
    with open(out_tif, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(f'downloaded -> {out_tif}')

meta = {
    'band_names': band_names, 'n_train': len(train_gdf), 'n_test': len(test_gdf),
    'dates': STACK5_DATES, 'best_block_m': best_block, 'cv_rmse_tune': best_rmse_tune,
    'best_params': best_params, 'holdout_metrics': holdout,
}
with open(f'{OUTPUT_DIR}/model_metadata_stack5_split.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print(f'\nSaved -> {OUTPUT_DIR}/model_metadata_stack5_split.json')

HOLDOUT (train-only model, scored on the held-out test partition):
  RMSE=17.98  MAE=13.49  R2=0.744  Bias=-0.03


submitted batch export task (id=MQH7EJX2EAQ3SRWKWTQXEZIM)


  RUNNING


  RUNNING


  RUNNING


...

  RUNNING


  RUNNING


  RUNNING


  COMPLETED


downloaded -> ./output/milan/ghsl/stack5_split/GHSL_predicted_RF_S2_stack5_split_Milan.tif

Saved -> ./output/milan/ghsl/stack5_split/model_metadata_stack5_split.json
